In [4]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import linear_kernel

In [5]:
df = df.dropna(subset=['title']).reset_index(drop=True)
text_columns = ['director', 'cast', 'listed_in', 'description', 'type']
for col in text_columns:
    df[col] = df[col].fillna('')

In [6]:
def create_soup(row):
    return f"{row['type']} {row['listed_in']} {row['director']} {row['cast']} {row['description']}"

df['metadata_soup'] = df.apply(create_soup, axis=1)

In [7]:
tfidf = TfidfVectorizer(stop_words='english', min_df=2, ngram_range=(1, 2))
tfidf_matrix = tfidf.fit_transform(df['metadata_soup'])

cosine_sim = linear_kernel(tfidf_matrix, tfidf_matrix)

indices = pd.Series(df.index, index=df['title'].str.lower()).drop_duplicates()

In [8]:
def recommend_movie(title, top_n=5):
    query_title = title.strip().lower()
    
    if query_title not in indices:
        partial_matches = df[df['title'].str.lower().str.contains(query_title, na=False)]['title'].values
        if len(partial_matches) > 0:
            print(f" '{title}' not found. Did you mean one of these?")
            for match in partial_matches[:3]:
                print(f"   -> {match}")
        else:
            print(f" Could not find '{title}' or any close matches in the database.")
        return None

    idx = indices[query_title]
    if isinstance(idx, pd.Series): 
        idx = idx.iloc[0]

    sim_scores = list(enumerate(cosine_sim[idx]))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    sim_scores = sim_scores[1:top_n + 1]
    
    movie_indices = [i[0] for i in sim_scores]
    match_percentages = [f"{round(i[1] * 100, 1)}%" for i in sim_scores]
    
    recommendations = df.iloc[movie_indices][['title', 'type', 'listed_in', 'description']].copy()
    recommendations['Match_Score'] = match_percentages
    
    return recommendations.reset_index(drop=True)

In [14]:
recommend_movie("Layla Majnun", top_n=3)

,title,type,listed_in,description,Match_Score
0,Kartini: Princess of Java,Movie,"Dramas, International Movies",A woman of nobility battles patriarchal norms ...,19.0%
1,Love Like the Falling Rain,Movie,"Dramas, International Movies, Romantic Movies","Fearing rejection, a young man struggles to de...",14.6%
2,Rudy Habibie,Movie,"Dramas, International Movies",A biopic that explores the hardships and trium...,14.1%
